In [ ]:
# Packages to load
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import zipfile
import matplotlib.pyplot as plt
import os
import glob

import urllib.request
from scipy.ndimage import gaussian_filter, minimum_filter
from matplotlib.colors import hsv_to_rgb

In [ ]:
!pip install -q py4DSTEM

# EDA(Data sourcing,Reading,Inspection & Virtual )

#First clean /kaggle/working/ to restore full 19 GB disk space
!rm -rf /kaggle/working/*

#Find the file in /kaggle/input (if added as Input) or download single 7.4 GB .npz
input_files = glob.glob("/kaggle/input/**/*.npz", recursive=True)
if input_files:
    file_path = input_files[0]
else:
    !kaggle datasets download -d mdramjanali/prefiltered-4dstem-au-nanoparticle-liquid-cell-tem -p /kaggle/working/ --unzip
    file_path = glob.glob("/kaggle/working/*.npz")[0]

print("Using dataset file:", file_path)

!rm -rf /kaggle/working/*

In [ ]:
!kaggle datasets download -d mdramjanali/prefiltered-4dstem-au-nanoparticle-liquid-cell-tem
!unzip -q prefiltered-4dstem-au-nanoparticle-liquid-cell-tem.zip -d /kaggle/working/
!rm prefiltered-4dstem-au-nanoparticle-liquid-cell-tem.zip

In [ ]:
npz_file = glob.glob("/kaggle/working/*.npz")[0]
print("File found at:", npz_file) 

In [ ]:
#Print all folders, subfolders, and files
for path, subdirs, files in os.walk('/kaggle/working'):
    print(path, subdirs, files)

#Load dataset and print keys
data = np.load('/kaggle/working/Prefiltered_4DSTEM_Au_nanoparticle_liquid_cell_TEM.npz')
print("Keys:", data.files)

In [ ]:
archive = np.load(npz_file)
data = archive[archive.files[0]]
del archive

In [ ]:
data.shape

In [ ]:
Ry, Rx, Ky, Kx = data.shape
print(f"Scan Grid: {Ry}x{Rx} | Detector: {Ky}x{Kx} | Dtype: {data.dtype}")

In [ ]:
cy, cx = Ky // 2, Kx // 2
yy, xx = np.ogrid[:Ky, :Kx]
r = np.hypot(xx - cx, yy - cy)
print(r)

In [ ]:
mean_dp = data.mean(axis=(0,1))
mean_dp

In [ ]:
plt.figure(figsize=(12, 7))
plt.imshow(np.log1p(mean_dp), cmap="inferno")
plt.title("Position-Averaged Convergent Beam Electron Diffraction (PACBED)")
plt.xlabel("Reciprocal Coordinate $K_x$ (detector pixels)")
plt.ylabel("Reciprocal Coordinate $K_y$ (detector pixels)")
plt.colorbar(label=r"Log Intensity $\ln(I + 1)$ (arb. units)")
plt.tight_layout()
plt.savefig('mf.png')
plt.show()

In [ ]:
plt.figure(figsize=(12, 7))
plt.imshow(data[256, 256], cmap="viridis", vmin=0, vmax=350)
plt.title("Electron Diffraction Pattern at Pixel (256, 256)")
plt.xlabel("Reciprocal Coordinate $K_x$ (detector pixels)")
plt.ylabel("Reciprocal Coordinate $K_y$ (detector pixels)")
plt.colorbar(label="Intensity (arb. units)")
plt.tight_layout()
#plt.savefig('dp.png')
plt.show()

from matplotlib import colormaps
list(colormaps)

In [ ]:
# Select two probe positions: center of particle vs surrounding liquid
dp_particle = data[Ry // 2, Rx // 2]
dp_liquid = data[0, 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(np.log1p(dp_particle), cmap="magma")
axes[0].set_title(f"Particle Center DP ({Ry//2}, {Rx//2})")
axes[0].set_xlabel(r"Reciprocal Coordinate $K_x$ (detector pixels)"); axes[0].set_ylabel(r"Reciprocal Coordinate $K_y$ (detector pixels)")

axes[1].imshow(np.log1p(dp_liquid), cmap="magma")
axes[1].set_title("Liquid Matrix DP (0, 0)")
axes[1].set_xlabel(r"Reciprocal Coordinate $K_x$ (detector pixels)"); axes[1].set_ylabel(r"Reciprocal Coordinate $K_y$ (detector pixels)")
plt.savefig('fdp.png')
plt.show()

In [ ]:
#real_space_view = sum(data[:, :, y, x] for y, x in zip(*np.where(r <= 12)))
real_space = data.sum(axis=(2, 3))

plt.figure(figsize=(12, 6), dpi=150)
im = plt.imshow(real_space, cmap="gray")
plt.xlim(0, 512)
plt.ylim(512, 0)  # Inverted so 0 is at the top, matching your reference
plt.title("Reconstructed Real-Space View (Gold Nanoparticles)", fontsize=12)
plt.xlabel("Real-Space Scan $R_x$ (scan pixels)")
plt.ylabel("Real-Space Scan $R_y$ (scan pixels)")

cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label("Aggregated Intensity", fontsize=10)
plt.savefig('rs.png')
plt.tight_layout()
plt.show()

In [ ]:
v_bf = sum(data[:, :, y, x] for y, x in zip(*np.where(r <= 12)))

plt.figure(figsize=(12, 7))
plt.imshow(v_bf, cmap="gray")
plt.title("Virtual Bright Field (V-BF) Image")
plt.xlabel("Real-Space Position $R_x$ (scan pixels)")
plt.ylabel("Real-Space Position $R_y$ (scan pixels)")
plt.colorbar(label="Transmitted Intensity (arb. units)")
plt.tight_layout()
plt.savefig('bf.png')
plt.show()

In [ ]:
v_adf = sum(data[:, :, y, x] for y, x in zip(*np.where((r > 18) & (r < 70))))

plt.figure(figsize=(12, 7))
plt.imshow(v_adf, cmap="viridis")
plt.title("Virtual Annular Dark Field (V-ADF) Image")
plt.xlabel("Real-Space Position $R_x$ (scan pixels)")
plt.ylabel("Real-Space Position $R_y$ (scan pixels)")
plt.colorbar(label="Scattered Intensity (arb. units)")
plt.tight_layout()
plt.savefig('df.png')
plt.show
#plt.savefig('C:\Users\mdram\Computational_Material-Science\4D STEM dataset of Au nanoparticle in liquid cell TEM/Plots/mf.png')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(np.log1p(mean_dp), cmap="inferno")
axes[0].set_title("1. Mean DP (Diffraction Space)")

axes[1].imshow(v_bf, cmap="gray")
axes[1].set_title("2. Virtual Bright Field (V-BF)")

axes[2].imshow(v_adf, cmap="viridis")
axes[2].set_title("3. Virtual Annular Dark Field (V-ADF)")

for ax in axes:
    ax.axis("off")
plt.savefig('combined.png')
plt.tight_layout()
plt.show()

INTERMEDIATE